# Аналіз та прогнозування фінансових часових рядів

## Мета роботи

Розробити систему прогнозування фінансових часових рядів на основі:
- Фрагментації на 3-тижневі вікна (21 день)
- Незалежної нормалізації кожного фрагменту
- Кластеризації методом самоорганізувальних карт (SOM / Kohonen)
- Прогнозування на основі типових патернів поведінки

## Основні параметри

- **Довжина фрагменту**: 21 день
- **Крок зміщення**: 7 днів
- **Довжина префіксу**: 14 днів (відома частина)
- **Горизонт прогнозу**: 7 днів (прогнозована частина)
- **Метод кластеризації**: SOM 5×6 (30 кластерів)
- **Джерело даних**: локальний CSV файл

## 1. Імпорт бібліотек

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Імпорт модулів проєкту
from src.data_loader import load_local_csv, validate_input_columns
from src.preprocessing import (
    prepare_time_series,
    create_fragments,
    split_train_experimental_series
)
from src.normalization import (
    normalize_fragments,
    normalize_single_fragment,
    validate_normalization
)
from src.clustering import (
    train_som,
    assign_clusters,
    get_som_weight_vectors,
    get_cluster_statistics,
    print_cluster_summary
)
from src.forecasting import forecast_next_week
from src.metrics import (
    calculate_all_metrics,
    save_metrics,
    save_metrics_txt,
    print_metrics_summary
)
from src.visualization import (
    plot_full_series,
    plot_raw_fragments,
    plot_normalized_fragments,
    plot_raw_vs_normalized_fragment,
    plot_cluster_fragments,
    plot_multiple_clusters,
    plot_som_heatmap,
    plot_target_cluster_match,
    plot_normalized_forecast,
    plot_real_forecast_vs_actual,
    plot_forecast_comparison_grid
)

print("✓ Усі модулі успішно імпортовано")

✓ Усі модулі успішно імпортовано


## 2. Завантаження даних

Завантажуємо локальний CSV файл з історичними даними про ціни BTC-USD.

In [ ]:
# Шлях до локального CSV файлу
DATA_PATH = '../data/raw/BTC-USD.csv'

# Завантаження даних
df = load_local_csv(DATA_PATH)

print(f"Завантажено {len(df)} рядків даних")
print(f"Період: {df['Date'].min()} - {df['Date'].max()}")
print(f"\nПерші 5 рядків:")
df.head()

Завантажено 1461 рядків даних
Період: 2019-01-01 00:00:00 - 2022-12-31 00:00:00

Перші 5 рядків:


,Date,Open,High,Low,Close,Adj Close,Volume
0,2019-01-01,3746.713379,3850.913818,3707.231201,3843.520020,3843.520020,4324200990
1,2019-01-02,3849.216309,3947.981201,3817.409424,3943.409424,3943.409424,5244856836
2,2019-01-03,3931.048584,3935.685059,3826.222900,3836.741211,3836.741211,4530215219
3,2019-01-04,3832.040039,3865.934570,3783.853760,3857.717529,3857.717529,4847965467
4,2019-01-05,3851.973877,3904.903076,3836.900146,3845.194580,3845.194580,5137609824


## 3. Візуалізація повного часового ряду

In [3]:
plot_full_series(
    df,
    value_column='Close',
    title='Повний часовий ряд BTC-USD',
    save_path='../results/figures/01_full_time_series.png'
)

Збережено: ../results/figures/01_full_time_series.png


## 4. Підготовка часового ряду

Очищення даних, видалення пропущених значень, вирівнювання на понеділок.

In [4]:
# Підготовка часового ряду
df_clean = prepare_time_series(df, align_to_monday=True)

print(f"Очищено: {len(df_clean)} рядків")
print(f"Період після очищення: {df_clean['Date'].min()} - {df_clean['Date'].max()}")
print(f"\nСтатистика:")
print(df_clean['Close'].describe())

Очищено: 1455 рядків
Період після очищення: 2019-01-07 00:00:00 - 2022-12-31 00:00:00

Статистика:
count     1455.000000
mean     23609.014772
std      17547.195942
min       3399.471680
25%       9172.597168
50%      17150.623047
75%      38740.271485
max      67566.828125
Name: Close, dtype: float64


## 5. Створення тренувальних та експериментальних даних

Останні 21 день використовуємо як експериментальне зображення:
- Перші 14 днів = префікс (відомі дані)
- Останні 7 днів = постфікс (для оцінки прогнозу)

In [5]:
# Розділення на тренувальні та експериментальні дані
split_data = split_train_experimental_series(
    df_clean,
    prefix_length=14,
    forecast_horizon=7
)

train_df = split_data['train_df']
experimental_full = split_data['experimental_full']
experimental_prefix = split_data['experimental_prefix']
actual_postfix = split_data['actual_postfix']

print(f"Тренувальні дані: {len(train_df)} рядків")
print(f"Експериментальний префікс: {len(experimental_prefix)} днів")
print(f"Фактичний постфікс: {len(actual_postfix)} днів")
print(f"\nТочка прогнозування: {split_data['forecast_start_date']}")
print(f"Кінець періоду прогнозу: {split_data['forecast_end_date']}")

Тренувальні дані: 1448 рядків
Експериментальний префікс: 14 днів
Фактичний постфікс: 7 днів

Точка прогнозування: 2022-12-25 00:00:00
Кінець періоду прогнозу: 2022-12-31 00:00:00


## 6. Створення фрагментів

Розбиваємо тренувальні дані на 3-тижневі фрагменти з кроком 7 днів.

In [6]:
# Створення фрагментів з тренувальних даних
X_raw, metadata = create_fragments(
    train_df,
    value_column='Close',
    fragment_length=21,
    step=7
)

print(f"Створено фрагментів: {X_raw.shape[0]}")
print(f"Розмір кожного фрагменту: {X_raw.shape[1]} днів")
print(f"\nПриклад метаданих фрагменту:")
metadata.head()

Створено фрагментів: 204
Розмір кожного фрагменту: 21 днів

Приклад метаданих фрагменту:


,fragment_index,start_date,end_date,start_position,end_position,x_min,x_max
0,0,2019-01-07,2019-01-27,0,20,3552.953125,4035.296387
1,1,2019-01-14,2019-02-03,7,27,3448.116943,3728.568359
2,2,2019-01-21,2019-02-10,14,34,3399.471680,3690.188232
3,3,2019-01-28,2019-02-17,21,41,3399.471680,3690.188232
4,4,2019-02-04,2019-02-24,28,48,3399.471680,4142.526855


## 7. Візуалізація сирих фрагментів

In [7]:
# Візуалізація декількох сирих фрагментів
plot_raw_fragments(
    X_raw,
    metadata,
    fragment_indices=[0, 50, 100, 150, 200, 250],
    title='Приклади сирих 3-тижневих фрагментів',
    save_path='../results/figures/02_raw_fragments.png'
)

Збережено: ../results/figures/02_raw_fragments.png


## 8. Нормалізація фрагментів

Кожен фрагмент нормалізується незалежно до діапазону [0, 1].

Формула: `x_norm = (x - x_min) / (x_max - x_min)`

In [8]:
# Нормалізація фрагментів
X_norm, x_mins, x_maxs = normalize_fragments(X_raw)

print(f"Форма нормалізованої матриці: {X_norm.shape}")
print(f"Діапазон значень: [{X_norm.min():.6f}, {X_norm.max():.6f}]")

# Валідація нормалізації
validate_normalization(X_raw, X_norm, x_mins, x_maxs)
print("✓ Нормалізація пройшла валідацію")

Форма нормалізованої матриці: (204, 21)
Діапазон значень: [0.000000, 1.000000]
✓ Нормалізація пройшла валідацію


## 9. Візуалізація нормалізованих фрагментів

In [9]:
# Візуалізація нормалізованих фрагментів
plot_normalized_fragments(
    X_norm,
    metadata,
    fragment_indices=[0, 50, 100, 150, 200, 250],
    title='Приклади нормалізованих фрагментів',
    save_path='../results/figures/03_normalized_fragments.png'
)

Збережено: ../results/figures/03_normalized_fragments.png


## 10. Порівняння сирого та нормалізованого фрагменту

In [10]:
# Порівняння сирого та нормалізованого фрагменту
plot_raw_vs_normalized_fragment(
    X_raw,
    X_norm,
    metadata,
    fragment_index=100,
    save_path='../results/figures/04_raw_vs_normalized.png'
)

Збережено: ../results/figures/04_raw_vs_normalized.png


## 11. Тренування SOM (самоорганізувальна карта)

Використовуємо мережу Кохонена для кластеризації нормалізованих фрагментів.

In [11]:
# Тренування SOM
som = train_som(
    X_norm,
    som_grid_x=5,
    som_grid_y=6,
    input_len=21,
    sigma=1.0,
    learning_rate=0.5,
    num_iterations=10000,
    random_seed=42
)

Тренування SOM:
  - Розмір сітки: 5 x 6 = 30 нейронів
  - Кількість фрагментів: 204
  - Довжина вхідного вектора: 21
  - Кількість ітерацій: 10000
  - Випадкове зерно: 42
  - Тренування SOM...
  ✓ Тренування завершено


## 12. Присвоєння кластерів

In [12]:
# Присвоєння кластерів
cluster_ids, neuron_coords, distances = assign_clusters(som, X_norm)

# Отримання вагових векторів
weight_matrix, neuron_mapping = get_som_weight_vectors(som)

print(f"Форма матриці вагових векторів: {weight_matrix.shape}")
print(f"Кількість кластерів: {len(neuron_mapping)}")

# Виведення підсумку кластеризації
print_cluster_summary(som, cluster_ids, weight_matrix)

Форма матриці вагових векторів: (30, 21)
Кількість кластерів: 30

ПІДСУМОК КЛАСТЕРИЗАЦІЇ SOM
Розмір сітки SOM: 5 x 6
Загальна кількість кластерів: 30
Загальна кількість фрагментів: 204
Непусті кластери: 30
Пусті кластери: 0

Найбільший кластер: 23 (18 фрагментів)
Найменший непустий кластер: 7 (3 фрагментів)

Розмір матриці вагових векторів: (30, 21)
Довжина кожного вагового вектора: 21



## 13. Візуалізація кластерів

In [13]:
# Карта щільності кластерів
plot_som_heatmap(
    cluster_ids,
    som_grid_x=5,
    som_grid_y=6,
    save_path='../results/figures/05_som_heatmap.png'
)

Збережено: ../results/figures/05_som_heatmap.png


In [14]:
# Знаходження найбільш населених кластерів
stats = get_cluster_statistics(cluster_ids, n_clusters=30)
top_clusters = np.argsort(stats['cluster_sizes'])[-6:][::-1]  # Top 6

print(f"Найбільші кластери: {top_clusters}")

# Візуалізація декількох кластерів
plot_multiple_clusters(
    X_norm,
    cluster_ids,
    weight_matrix,
    cluster_list=top_clusters.tolist(),
    neuron_mapping=neuron_mapping,
    title='Топ-6 найбільших кластерів',
    save_path='../results/figures/06_top_clusters.png'
)

Найбільші кластери: [23 14  1 28 15  9]
Збережено: ../results/figures/06_top_clusters.png


## 14. Прогнозування наступних 7 днів

Використовуємо експериментальний префікс для знаходження цільового кластера та побудови прогнозу.

In [15]:
# Підготовка експериментального префіксу
experimental_prefix_raw = experimental_prefix['Close'].values
actual_postfix_values = actual_postfix['Close'].values

print(f"Експериментальний префікс: {len(experimental_prefix_raw)} днів")
print(f"Діапазон префіксу: [{experimental_prefix_raw.min():.2f}, {experimental_prefix_raw.max():.2f}]")

# Виконання прогнозування
forecast_result = forecast_next_week(
    experimental_prefix_raw=experimental_prefix_raw,
    weight_matrix=weight_matrix,
    neuron_mapping=neuron_mapping
)

# Розпакування результатів
target_cluster_id = forecast_result['cluster_id']
target_weight_vector = forecast_result['target_weight_vector']
forecast_norm = forecast_result['forecast_norm']
forecast_real = forecast_result['forecast_real']
prefix_norm = forecast_result['prefix_norm']

print(f"\nПрогноз на наступні 7 днів:")
for i, value in enumerate(forecast_real, 1):
    print(f"  День {i}: {value:.2f} USD")

Експериментальний префікс: 14 днів
Діапазон префіксу: [16439.68, 17815.65]
ПРОГНОЗУВАННЯ НАСТУПНИХ 7 ДНІВ

Пошук цільового кластера:
  - Кількість кластерів для порівняння: 30
  - Довжина експериментального префіксу: 14
  - Префікс нормалізовано: min=16439.68, max=17815.65
  - Цільовий кластер знайдено: 21
  - Відстань до цільового кластера: 0.800885
  - Координати нейрона: (1, 4)

Побудова прогнозу:
  - Довжина нормалізованого прогнозу: 7
  - Діапазон значень: [0.2827, 0.8006]

Денормалізація прогнозу:
  - Використано масштаб префіксу: [16439.68, 17815.65]
  - Діапазон реальних цін прогнозу: [16828.73, 17541.33]
ПРОГНОЗУВАННЯ ЗАВЕРШЕНО


Прогноз на наступні 7 днів:
  День 1: 16884.92 USD
  День 2: 16828.73 USD
  День 3: 16995.32 USD
  День 4: 16979.17 USD
  День 5: 17436.43 USD
  День 6: 17415.37 USD
  День 7: 17541.33 USD


## 15. Візуалізація відповідності цільового кластера

In [16]:
# Візуалізація відповідності префіксу та цільового кластера
plot_target_cluster_match(
    experimental_prefix_norm=prefix_norm,
    target_weight_vector=target_weight_vector,
    cluster_id=target_cluster_id,
    neuron_coords=forecast_result['neuron_coords'],
    distance=forecast_result['distance'],
    save_path='../results/figures/07_target_cluster_match.png'
)

Збережено: ../results/figures/07_target_cluster_match.png


## 16. Візуалізація нормалізованого прогнозу

In [17]:
# Візуалізація нормалізованого прогнозу
plot_normalized_forecast(
    experimental_prefix_norm=prefix_norm,
    forecast_norm=forecast_norm,
    target_weight_vector=target_weight_vector,
    cluster_id=target_cluster_id,
    save_path='../results/figures/08_normalized_forecast.png'
)

Збережено: ../results/figures/08_normalized_forecast.png


## 17. Обчислення метрик якості прогнозу

In [18]:
# Обчислення метрик
metrics = calculate_all_metrics(
    y_true=actual_postfix_values,
    y_pred=forecast_real
)

# Виведення метрик
print_metrics_summary(
    metrics,
    y_true=actual_postfix_values,
    y_pred=forecast_real
)


МЕТРИКИ ЯКОСТІ ПРОГНОЗУ

MAE (Mean Absolute Error):               491.3504
RMSE (Root Mean Square Error):           603.3691
MAPE (Mean Absolute Percentage Error):   2.9582%

Кількість точок прогнозу: 7
Середнє фактичне значення: 16689.14
Середнє прогнозоване значення: 17154.47

Інтерпретація:
  ✓ Дуже точний прогноз (MAPE < 10%)



## 18. Збереження метрик

In [19]:
# Додаткова інформація для збереження
additional_info = {
    'forecast_start_date': str(split_data['forecast_start_date']),
    'forecast_end_date': str(split_data['forecast_end_date']),
    'target_cluster_id': int(target_cluster_id),
    'distance_to_cluster': float(forecast_result['distance']),
    'n_training_fragments': int(X_raw.shape[0])
}

# Збереження метрик у JSON
save_metrics(
    metrics,
    save_path='../results/metrics/forecast_metrics.json',
    additional_info=additional_info
)

# Збереження метрик у текстовий файл
save_metrics_txt(
    metrics,
    save_path='../results/metrics/forecast_metrics.txt',
    y_true=actual_postfix_values,
    y_pred=forecast_real
)

Метрики збережено: ../results/metrics/forecast_metrics.json
Метрики збережено (текст): ../results/metrics/forecast_metrics.txt


## 19. Візуалізація прогнозу проти фактичних значень

In [20]:
# Основний графік результатів
plot_real_forecast_vs_actual(
    experimental_prefix_raw=experimental_prefix_raw,
    forecast_real=forecast_real,
    actual_postfix=actual_postfix_values,
    prefix_dates=experimental_prefix['Date'],
    postfix_dates=actual_postfix['Date'],
    metrics=metrics,
    save_path='../results/figures/09_forecast_vs_actual.png'
)

Збережено: ../results/figures/09_forecast_vs_actual.png


## 20. Комплексна візуалізація (нормалізований + реальний простір)

In [21]:
# Створення комплексної візуалізації
plot_forecast_comparison_grid(
    experimental_prefix_raw=experimental_prefix_raw,
    forecast_real=forecast_real,
    actual_postfix=actual_postfix_values,
    prefix_norm=prefix_norm,
    forecast_norm=forecast_norm,
    target_weight_vector=target_weight_vector,
    prefix_dates=experimental_prefix['Date'],
    postfix_dates=actual_postfix['Date'],
    cluster_id=target_cluster_id,
    metrics=metrics,
    save_path='../results/figures/10_forecast_comparison_grid.png'
)

Збережено: ../results/figures/10_forecast_comparison_grid.png


## 21. Підсумкова таблиця результатів

In [22]:
# Створення таблиці порівняння
comparison_df = pd.DataFrame({
    'Дата': actual_postfix['Date'].values,
    'Фактична ціна': actual_postfix_values,
    'Прогноз': forecast_real,
    'Абсолютна помилка': np.abs(actual_postfix_values - forecast_real),
    'Відносна помилка (%)': 100 * np.abs(actual_postfix_values - forecast_real) / actual_postfix_values
})

print("\nДетальне порівняння прогнозу та фактичних значень:")
print("="*80)
comparison_df


Детальне порівняння прогнозу та фактичних значень:


,Дата,Фактична ціна,Прогноз,Абсолютна помилка,Відносна помилка (%)
0,2022-12-25,16841.986328,16884.920677,42.934349,0.254924
1,2022-12-26,16919.804688,16828.729802,91.074886,0.538274
2,2022-12-27,16717.173828,16995.320440,278.146612,1.663838
3,2022-12-28,16552.572266,16979.167146,426.594880,2.577212
4,2022-12-29,16642.341797,17436.425311,794.083514,4.771465
5,2022-12-30,16602.585938,17415.372225,812.786287,4.895540
6,2022-12-31,16547.496094,17541.328533,993.832439,6.005938


## 22. Висновки

### Методологія

У цій роботі було реалізовано повний конвеєр аналізу та прогнозування фінансових часових рядів:

1. **Фрагментація**: Часовий ряд розбито на перекриваючі 3-тижневі фрагменти з кроком 1 тиждень
2. **Нормалізація**: Кожен фрагмент нормалізовано незалежно до діапазону [0, 1]
3. **Кластеризація**: Використано самоорганізувальну карту (SOM 5×6) для групування схожих патернів
4. **Прогнозування**: На основі відповідності 14-денного префіксу знайдено цільовий кластер і використано його постфікс як прогноз
5. **Денормалізація**: Прогноз переведено назад до реального цінового масштабу

### Результати

- **Кількість тренувальних фрагментів**: {X_raw.shape[0]}
- **Кількість кластерів**: 30
- **Цільовий кластер**: {target_cluster_id}
- **Відстань до цільового кластера**: {forecast_result['distance']:.6f}

### Метрики якості

- **MAE (Mean Absolute Error)**: {metrics['MAE']:.2f} USD
- **RMSE (Root Mean Square Error)**: {metrics['RMSE']:.2f} USD
- **MAPE (Mean Absolute Percentage Error)**: {metrics['MAPE']:.2f}%

### Інтерпретація

Отримані результати демонструють {'дуже точний' if metrics['MAPE'] < 10 else 'хороший' if metrics['MAPE'] < 20 else 'прийнятний' if metrics['MAPE'] < 50 else 'неточний'} прогноз.

Метод SOM-кластеризації успішно виявляє типові патерни поведінки часового ряду та дозволяє використовувати їх для прогнозування майбутніх значень на основі схожості з історичними даними.

### Переваги підходу

1. **Інтерпретовність**: Прогноз базується на реальних історичних патернах
2. **Масштабонезалежність**: Нормалізація дозволяє порівнювати патерни за різних цінових рівнів
3. **Відсутність перенавчання**: Не потребує налаштування великої кількості параметрів
4. **Локальна робота**: Не потребує доступу до інтернету

### Обмеження

1. Прогноз базується на припущенні, що майбутнє поводитиметься схоже до минулого
2. Непередбачені події (новини, регуляції) можуть порушити патерни
3. Горизонт прогнозу обмежений 7 днями

### Можливі покращення

1. Експериментування з різними розмірами SOM-сітки
2. Використання додаткових ознак (Volume, Open, High, Low)
3. Ансамблювання прогнозів від кількох найближчих кластерів
4. Адаптивне оновлення моделі з надходженням нових даних

In [23]:
print("\n" + "="*80)
print("ЛАБОРАТОРНУ РОБОТУ ЗАВЕРШЕНО УСПІШНО")
print("="*80)
print(f"\nУсі результати збережено у папки:")
print("  - results/figures/  (графіки)")
print("  - results/metrics/  (метрики)")
print("\n✓ Notebook виконано повністю")


ЛАБОРАТОРНУ РОБОТУ ЗАВЕРШЕНО УСПІШНО

Усі результати збережено у папки:
  - results/figures/  (графіки)
  - results/metrics/  (метрики)

✓ Notebook виконано повністю
